## Mesure de l'observabilité : 
Vecteur X : X = [θ1 - θ2, ω1, ω2, Tm1, Tm2, N]

Vecteur d'entrées U : U = [PG1, PG2, PL1, PL2]   

Vecteur de sortie Y : Y = [F12, F21, PG1, PG2, PL1, PL2]

$$\dot{X} = AX + BU $$
$$Y = CX + DU$$

In [46]:
using LinearAlgebra

## Paramètres de départ

In [47]:
KL = 3064 # 1 ligne
ω0 = 100*pi
P01 = 600
P02 = 400
Pmin1 = 0
Pmin2 = 0
Pmax1 = 1000
Pmax2 = 600
Pr1 = 100
Pr2 = 50
J1 = 0.4
J2 = 0.1
D1 = 0.04
D2 = 0.02
α1 = 100
α2 = 100
β1 = 2000
β2 = 2000
Ks =0.05;

### On calcule la dérivée temporelle de X

$$\dot{\theta}_1 - \dot{\theta}_2 = \omega_1 - \omega_2$$

$$J_1 \dot{\omega}_1 = T_1^m - \frac{P_1^G}{\omega_1} - D_1(\omega_1 - \omega_0)$$

$$J_2 \dot{\omega}_2 = T_2^m - \frac{P_2^G}{\omega_2} - D_2(\omega_2 - \omega_0)$$

$$\dot{T}_1^m = -\alpha_1(P_1^m - P_1^c) - \beta_1(\omega_1 - \omega_0)$$

$$\dot{T}_2^m = -\alpha_2(P_2^m - P_2^c) - \beta_2(\omega_2 - \omega_0)$$

$$\dot{N} = K_s\left(\frac{J_1\omega_1 + J_2\omega_2}{J_1+J_2} - \omega_0\right)$$

### On linéarise en considérant ω1≈ω2≈ω0 et P1c≈P01+NPr1, P2c≈P02+NPr2



In [ ]:
A = [
    0   1       -1  0   0   0;
    0   -D1/J1  0   1/J1    0   0;
    0   0   -D2/J2  0   1/J2    0;
    0   -β1     0   (-α1 *ω0)   0   -α1*Pr1;
    0   0       -β2     0   (-α2 *ω0)   -α2*Pr2;
    0  (Ks * J1)/(J1 + J2)   (Ks * J2)/(J1 + J2)  0    0   0
    ]

6×6 Matrix{Float64}:
 0.0     -0.1             0.2        …  -10.0             0.0
 0.0  -4999.99            0.0             0.0        -25000.0
 0.0      0.0        -20000.0            -3.14161e5  -50000.0
 0.0      6.28317e7    -100.0             0.0             3.14159e8
 0.0   -200.0             6.28322e7       9.8694e8        1.5708e8
 0.0     -0.004          -0.002      …    0.1             0.0

In [49]:
rank(A)

5

## Matrice C

In [50]:
C = [
    KL 0 0 0 0 0;
    -KL 0 0 0 0 0;
    KL 0 0 0 0 0;
    -KL 0 0 0 0 0;
    0 0 0 0 0 0;
    0 0 0 0 0 0;
    ]

6×6 Matrix{Int64}:
  3064  0  0  0  0  0
 -3064  0  0  0  0  0
  3064  0  0  0  0  0
 -3064  0  0  0  0  0
     0  0  0  0  0  0
     0  0  0  0  0  0

Matrice O = [C CA CA² ... CA^5]^T

In [51]:
O = vcat([C * (A^k) for k in 0:5]...)   # concaténation horizontale

36×6 Matrix{Float64}:
  3064.0      0.0             0.0          0.0          0.0        0.0
 -3064.0      0.0             0.0          0.0          0.0        0.0
  3064.0      0.0             0.0          0.0          0.0        0.0
 -3064.0      0.0             0.0          0.0          0.0        0.0
     0.0      0.0             0.0          0.0          0.0        0.0
     0.0      0.0             0.0          0.0          0.0        0.0
     0.0   3064.0         -3064.0          0.0          0.0        0.0
     0.0  -3064.0          3064.0          0.0          0.0        0.0
     0.0   3064.0         -3064.0          0.0          0.0        0.0
     0.0  -3064.0          3064.0          0.0          0.0        0.0
     ⋮                                                             ⋮
     0.0     -4.81298e11      1.92519e12  -7.5601e12    3.024e13   2.40648e12
     0.0      0.0             0.0          0.0          0.0        0.0
     0.0      0.0             0.0          0.0    

Calcul du rang de O

In [52]:
rank(O)

5